# Driver behaviour: clocks, plug probability and the shared skip share

How one EV's daily clocks are drawn, how the six source cohorts differ, and how a run turns
"would this EV plug in tonight" into a fleet-wide, correlated outcome instead of 1,000 independent
coin flips. Every value below is `evidence_kind="illustrative"` or `"illustrative_synthetic"`
(`docs/MODEL_CONTRACT.md`); nothing here is observed telemetry, a settlement or a tariff.

**The idea in one paragraph.** Each EV-night has two independent clocks — an evening plug-in and
the next morning's home departure — drawn as truncated Student-t shifts around its cohort's usual
time (decision 0004 item 51). An EV unplugs only on a morning it drives: on a day with no trip it
stays plugged in until its next driving day, so a session can span several nights (item 68).
Whether an EV plugs in at all is its cohort's source plug
probability, cut by a **shared skip share**: one logit shock per simulated day and one per
simulated week move every clocked EV's odds together, so a "quiet night" or a "the source rate was
itself an estimate" event is shared, not independent (decision 0004 items 59, 60, 63 and 64;
`docs/contracts/trading-events-v1.md` §10.0–§10.1). Four illustrative charger manufacturers add a
second shared source: a manufacturer's cloud outage removes its whole slice of the fleet from
control for a night. Both mechanisms make the fleet's deliverable plug-in count **not** shrink as
`1/√N` the way independent coin flips would — the closing section shows that directly.

Read `docs/decisions/0004-explicit-action-and-horizon-choices.md` items 51, 59, 60, 63 and 64, and
`docs/contracts/trading-events-v1.md` §10.0–§10.1, for the full arithmetic behind the numbers this
notebook reads live rather than restates.


In [1]:
# --- MODEL IMPORTS (one cell) -----------------------------------------------
# Everything the notebook imports from axle_studio lives in this one cell, so a module rename is
# a one-cell change; no other cell imports from axle_studio directly.
from dataclasses import replace
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from axle_studio.model import assumptions, clock, physics, sampling
from axle_studio.model.assumptions import MANUFACTURER_IDS, MANUFACTURER_LABELS
from axle_studio.model.forecast import simulate_forecast
from axle_studio.ui.style import ARCHETYPE_COLOURS, CHART_HEIGHTS, apply_notebook_style

# "notebook_connected" embeds only the figure data and loads plotly.js from a CDN, which keeps the
# committed notebook small. GitHub does not run that script, so every chart is paired with a plain
# text table (decision 0004 item 29).
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 120)
LONDON = "Europe/London"

## Try it

The run below is small (50 EVs, 5 simulated weeks) so the whole notebook executes in well under
30 seconds; raise `VEHICLE_COUNT` and `WORLD_COUNT` for steadier numbers, at the cost of runtime.
`START_LOCAL_DATE` is a Monday, so the bank-holiday-Monday preset of section 3 has a date to act
on.


In [2]:
START_LOCAL_DATE = date(2026, 1, 12)  # the London date the seven study nights start (a Monday)
VEHICLE_COUNT = 50
WORLD_COUNT = 5  # simulated weeks
SEED = int(assumptions.SIMULATION["seed"].value)

VALUES = {"vehicle_count": VEHICLE_COUNT, "evaluation_world_count": WORLD_COUNT, "seed": SEED}
resolved = assumptions.resolve_values(VALUES)
settings = assumptions.run_settings(resolved, START_LOCAL_DATE)
fixture = assumptions.cohort_fixture(resolved)
inputs = assumptions.forecast_inputs(
    resolved, warmup_days=settings.warmup_days, study_days=settings.study_days
)

# The action model's own draws (§10.0 call order): population, then departures, trips, the
# shared plug-in factors, connections and weather; then prices, non-response and manufacturer
# outages. One run gives every section below a consistent population and set of simulated weeks.
BASE = simulate_forecast(settings, fixture, model="action", **inputs)
pd.DataFrame(
    [
        {
            "vehicles": settings.vehicle_count,
            "simulated weeks": settings.evaluation_world_count,
            "warm-up days": settings.warmup_days,
            "study days": settings.study_days,
            "sampled London dates": settings.sampled_day_count,
            "seed": settings.seed,
        }
    ]
)

,vehicles,simulated weeks,warm-up days,study days,sampled London dates,seed
0,50,5,7,7,15,42


## 1. The six cohorts

`assumptions.cohort_fixture()` builds the six source archetypes the population is drawn from
(crosswalk-reviewed against the source workbook); `sampling.build_population` then draws exact
counts by **largest remainder**, not by chance — a cohort's count is a deterministic function of
its share and the fleet size, not a coin flip. That matters here: `always_plugged_in` is a 1 %
share, and at a fleet of a few dozen EVs its exact count can be zero. A cohort table read from the
fixture would still show its plug probability and clock hours; only a **population** built at this
fleet size can show whether it actually has any EVs this run. Never assume a small notebook run
has every cohort represented — read the count.


In [3]:
cohort_table = pd.DataFrame(
    [
        {
            "cohort_id": c.cohort_id,
            "source_name": c.source_name,
            "share_%": round(100 * c.population_share_fraction, 1),
            "plug_probability": c.plug_probability,
            "weekday_arrival": f"{c.arrival_local_hour:02d}:00",
            "weekday_departure": f"{c.departure_local_hour:02d}:00",
            "has_weekend_window": c.cohort_id in assumptions.WEEKEND_WINDOW_COHORTS,
        }
        for c in fixture.cohorts
    ]
)
cohort_table

,cohort_id,source_name,share_%,plug_probability,weekday_arrival,weekday_departure,has_weekend_window
0,average_uk,Average (UK),40.0,1.0,18:00,07:00,True
1,intelligent_octopus,Intelligent Octopus average,30.0,1.0,18:00,07:00,True
2,infrequent_charging,Infrequent charging,10.0,0.2,18:00,07:00,False
3,infrequent_driving,Infrequent driving,10.0,1.0,18:00,07:00,False
4,scheduled_charging,Scheduled charging,9.0,1.0,22:00,09:00,False
5,always_plugged_in,Always plugged-in,1.0,1.0,15:00,14:00,False


In [4]:
cohort_order = [c.cohort_id for c in fixture.cohorts]

# The run's own population (drawn first, before any world is sampled).
base_counts = BASE.units["cohort_id"].value_counts().reindex(cohort_order, fill_value=0)

# The same fixture, a bigger toy fleet, so the reader can see the largest-remainder rule give the
# 1 %-share cohort a non-zero count elsewhere in this notebook (used again in section 3).
TOY_VEHICLE_COUNT = 200
toy_settings = replace(settings, vehicle_count=TOY_VEHICLE_COUNT)
toy_rng = np.random.default_rng(SEED)
TOY_UNITS = sampling.build_population(
    toy_settings,
    fixture,
    toy_rng,
    daily_miles_cv_by_cohort=inputs["daily_miles_cv_by_cohort"],
    personal_mileage_cv_by_cohort=inputs["personal_mileage_cv_by_cohort"],
)
toy_counts = TOY_UNITS["cohort_id"].value_counts().reindex(cohort_order, fill_value=0)

count_table = pd.DataFrame(
    {f"count at N={VEHICLE_COUNT}": base_counts, f"count at N={TOY_VEHICLE_COUNT}": toy_counts}
)
assert int(count_table[f"count at N={VEHICLE_COUNT}"].sum()) == VEHICLE_COUNT
assert int(count_table[f"count at N={TOY_VEHICLE_COUNT}"].sum()) == TOY_VEHICLE_COUNT
zero_at_base = count_table.index[count_table[f"count at N={VEHICLE_COUNT}"] == 0].tolist()
zero_label = ", ".join(zero_at_base) if zero_at_base else "no cohort"
print(
    f"At N={VEHICLE_COUNT}, {zero_label} drew zero EVs this run "
    f"(largest-remainder counts, not chance); every cohort has at least one EV at "
    f"N={TOY_VEHICLE_COUNT}."
)
count_table

At N=50, always_plugged_in drew zero EVs this run (largest-remainder counts, not chance); every cohort has at least one EV at N=200.


,count at N=50,count at N=200
cohort_id,,
average_uk,20,80
intelligent_octopus,15,60
infrequent_charging,5,20
infrequent_driving,5,20
scheduled_charging,5,18
always_plugged_in,0,2


## 2. Departure and plug-in clocks

Decision 0004 item 51: each EV-night has two **independent** clocks. The home departure — which is
the trip's start on a driving day and the unplug instant (item 68: only on a driving day; with no
trip the EV stays plugged in) — is one truncated Student-t shift
around the cohort's usual departure hour; the evening plug-in is a second, independent shift around
its usual arrival hour. Both use the same construction (`sampling.truncated_t_shift_minutes`):
`t(df).ppf(lo + U·(hi−lo))·scale`, rounded to the half hour, truncated (not clipped) at the bound
below, so the tail mass spreads inside the bound instead of piling up exactly on it.


In [5]:
clip_minutes = float(inputs["trip_behaviour"]["clock_clip_minutes"])
t_df = float(inputs["trip_behaviour"]["clock_t_df"])
by_cohort = inputs["trip_behaviour"]["by_cohort"]

clock_table = (
    pd.DataFrame(
        [
            {
                "cohort_id": cohort_id,
                "departure_scale_min": scales["departure_scale_minutes"],
                "plug_in_scale_min (weekday)": scales["plug_in_scale_minutes"],
                "plug_in_scale_min (weekend)": scales["weekend_plug_in_scale_minutes"],
            }
            for cohort_id, scales in by_cohort.items()
        ]
    )
    .set_index("cohort_id")
    .loc[list(cohort_order)]
)
print(f"clock_t_df={t_df:g}  clock_clip_minutes={clip_minutes:g}  (fleet-wide, both clocks)")
clock_table

clock_t_df=4  clock_clip_minutes=180  (fleet-wide, both clocks)


,departure_scale_min,plug_in_scale_min (weekday),plug_in_scale_min (weekend)
cohort_id,,,
average_uk,75.0,60.0,75.0
intelligent_octopus,75.0,60.0,75.0
infrequent_charging,75.0,60.0,60.0
infrequent_driving,90.0,60.0,60.0
scheduled_charging,30.0,20.0,20.0
always_plugged_in,75.0,0.0,0.0


### Plug-out is departure: one draw, two uses

`sampling.sample_daily_trip_inputs` takes the *same* departure array `sampling.sample_departure_times`
draws and uses it both to end the previous night's home session and, on a driving day, as the trip's
start instant — never a second draw. The check below finds every (world, date, EV) cell where the
EV drives on date `d` and the session that started on date `d − 1` was accepted, and confirms the two
recorded instants are exactly equal.


In [6]:
ev_inputs = BASE.evaluation.inputs
drives = ev_inputs["drives_today"]
trip_departure = ev_inputs["trip_departure_utc"]
connection_end = ev_inputs["connection_end_utc"]
accepted = ev_inputs["connection_session_accepted"]

# date d's trip start vs. the departure that ended date (d - 1)'s session: decision 51 says
# these are the same recorded instant, not two draws that happen to agree.
driving_with_prior_session = drives[:, 1:, :] & accepted[:, :-1, :]
paired_count = int(driving_with_prior_session.sum())
assert paired_count > 0, "no driving day with a prior accepted session at this fleet size"
matches = (
    trip_departure[:, 1:, :][driving_with_prior_session]
    == connection_end[:, :-1, :][driving_with_prior_session]
)
assert bool(matches.all()), (
    "a driving day's trip start must equal the departure ending the prior session"
)
print(
    f"{paired_count} (world, date, EV) cells checked: every driving day's trip start equals the "
    "departure ending the previous night's session, exactly."
)

1808 (world, date, EV) cells checked: every driving day's trip start equals the departure ending the previous night's session, exactly.


### The shift shape, per cohort

A dedicated draw (not tied to the run above) shows the truncated-t shape itself: fat tails around
each cohort's usual departure time, a hard ±`clock_clip_minutes` bound, and 30-minute rounding.
`scheduled_charging`'s tighter scale and `infrequent_driving`'s wider one (plan 2026-09-29 A1) show
up directly as narrower or wider histograms.


In [7]:
shift_rng = np.random.default_rng(SEED)
SHIFT_DRAWS = 20_000
# Pre-binned (not a raw-point go.Histogram): every chart in this notebook family ships its data as
# a small aggregate, paired with the table below, rather than embedding 20,000 points per panel.
bin_edges = np.arange(-clip_minutes - 15.0, clip_minutes + 30.0, 30.0)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2.0

figure = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[fixture.cohorts[i].source_name for i in range(6)],
    shared_yaxes=True,
)
shift_summary = []
bin_table = {}
for index, cohort_id in enumerate(cohort_order):
    scale = by_cohort[cohort_id]["departure_scale_minutes"]
    uniforms = shift_rng.random(SHIFT_DRAWS)
    shifts = sampling.truncated_t_shift_minutes(uniforms, np.asarray(scale), t_df, clip_minutes)
    assert shifts.min() >= -clip_minutes - 1e-9 and shifts.max() <= clip_minutes + 1e-9
    assert np.all(np.mod(shifts, 30.0) == 0.0), "every shift must round to the half hour"
    counts, _ = np.histogram(shifts, bins=bin_edges)
    bin_table[cohort_id] = counts
    row, col = index // 3 + 1, index % 3 + 1
    figure.add_trace(
        go.Bar(
            x=bin_centres,
            y=100 * counts / SHIFT_DRAWS,
            marker_color=ARCHETYPE_COLOURS[index],
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    shift_summary.append(
        {
            "cohort_id": cohort_id,
            "scale_min": scale,
            "p05_min": float(np.percentile(shifts, 5)),
            "p95_min": float(np.percentile(shifts, 95)),
            "share_within_2h_%": round(100 * float((np.abs(shifts) <= 120).mean()), 1),
        }
    )
figure.update_layout(
    title_text=f"Departure clock shift (minutes), {SHIFT_DRAWS:,} draws per cohort"
)
apply_notebook_style(figure, height=CHART_HEIGHTS["small_multiple_panel"] * 2 + 80)

In [8]:
pd.DataFrame(shift_summary).set_index("cohort_id")

,scale_min,p05_min,p95_min,share_within_2h_%
cohort_id,,,,
average_uk,75.0,-120.0,120.0,92.3
intelligent_octopus,75.0,-120.0,120.0,92.0
infrequent_charging,75.0,-120.0,120.0,92.2
infrequent_driving,90.0,-150.0,150.0,89.7
scheduled_charging,30.0,-60.0,60.0,99.3
always_plugged_in,75.0,-120.0,120.0,92.1


### Plug-in and departure are drawn independently

The plug-in shift and the departure shift are two separate blocks of uniforms, drawn in a fixed
order regardless of parameter values (common random numbers, plan M6). Drawing both from one
generator, in sequence, and correlating them the way the run itself would consume the generator
shows the independence directly: the correlation should sit near zero, not merely "small by
construction".


In [9]:
independence_rng = np.random.default_rng(SEED)
CHECK_DRAWS = 5_000
departure_scale_check = by_cohort["average_uk"]["departure_scale_minutes"]
plug_in_scale_check = by_cohort["average_uk"]["plug_in_scale_minutes"]

departure_shift_check = sampling.truncated_t_shift_minutes(
    independence_rng.random(CHECK_DRAWS), np.asarray(departure_scale_check), t_df, clip_minutes
)
plug_in_shift_check = sampling.truncated_t_shift_minutes(
    independence_rng.random(CHECK_DRAWS), np.asarray(plug_in_scale_check), t_df, clip_minutes
)
correlation = float(np.corrcoef(departure_shift_check, plug_in_shift_check)[0, 1])
# Sampling SE of a correlation at this draw count is about 1/sqrt(CHECK_DRAWS) ~ 0.014; 0.08 is a
# generous, still genuinely failable, bound.
assert abs(correlation) < 0.08, (
    f"plug-in and departure shifts should be independent, got {correlation:.3f}"
)
print(f"correlation(departure shift, plug-in shift) = {correlation:.4f} over {CHECK_DRAWS} draws")

correlation(departure shift, plug-in shift) = 0.0140 over 5000 draws


## 3. Trips, plug probability and the shared skip share

Each cohort's daily trip behaviour (`assumptions.TRIP_BEHAVIOUR`) sets how often it drives on a
weekday or weekend and how long it dwells away; combined with the cohort's average daily miles,
this is the energy a session must recover before the next departure. The source plug
probabilities in the cohort table above are "would plug in" rates: five of the six cohorts plug in
with probability 1.0, so a shock could not move them directly (`logit(1) = +∞`). Decision 0004 item
60 resolves this with a **skip share** — the fraction of would-be sessions that do not happen on a
given night — shared by every clocked EV in a simulated world that night.


In [10]:
trip_table = pd.DataFrame(
    [
        {
            "cohort_id": c.cohort_id,
            "daily_miles_mean": round(c.daily_miles_mean, 1),
            "weekday_drive_probability": inputs["trip_behaviour"]["by_cohort"][c.cohort_id][
                "weekday_drive_probability"
            ],
            "weekend_drive_probability": inputs["trip_behaviour"]["by_cohort"][c.cohort_id][
                "weekend_drive_probability"
            ],
            "destination_dwell_min": inputs["trip_behaviour"]["by_cohort"][c.cohort_id][
                "destination_dwell_minutes"
            ],
        }
        for c in fixture.cohorts
    ]
).set_index("cohort_id")
trip_table

,daily_miles_mean,weekday_drive_probability,weekend_drive_probability,destination_dwell_min
cohort_id,,,,
average_uk,25.8,0.85,0.75,480
intelligent_octopus,77.0,0.85,0.75,480
infrequent_charging,25.8,0.85,0.75,480
infrequent_driving,15.6,0.35,0.45,180
scheduled_charging,25.8,0.85,0.75,480
always_plugged_in,25.8,0.20,0.35,120


In [11]:
factor_values = assumptions.shared_factor_inputs()
skip_median = factor_values["behaviour.plug_in_skip_median"]
day_sd = factor_values["behaviour.plug_in_day_factor_sd"]
week_sd = factor_values["behaviour.plug_in_week_factor_sd"]
print(
    f"plug_in_skip_median={skip_median:.2f}  plug_in_day_factor_sd={day_sd:.2f}  "
    f"plug_in_week_factor_sd={week_sd:.2f}  (read live from assumptions.SHARED_FACTORS)"
)

# logit q = logit(median) + week_sd . y_w + day_sd . z_wd + holiday shift; p_eff = p . (1 - q).
week_factor, day_factor, skip_share = (
    BASE.plug_in_factors["week"],
    BASE.plug_in_factors["day"],
    BASE.plug_in_factors["skip_share"],
)
# Sampled date index d = warmup_days is start_local_date itself (§10.1a notation): the study
# evenings are the next settings.study_days indices, in order.
warmup_day_count = settings.warmup_days
study_dates = [settings.start_local_date + timedelta(days=k) for k in range(settings.study_days)]
study_skip_share = skip_share[:, warmup_day_count : warmup_day_count + settings.study_days]

skip_figure = go.Figure()
for world_id in range(settings.evaluation_world_count):
    skip_figure.add_trace(
        go.Scatter(
            x=study_dates,
            y=100 * study_skip_share[world_id],
            mode="lines+markers",
            name=f"week {world_id}",
            line={"color": ARCHETYPE_COLOURS[world_id % len(ARCHETYPE_COLOURS)], "width": 1.5},
        )
    )
skip_figure.update_layout(
    title_text="Shared skip share by study night, one line per simulated week",
    yaxis_title="skip share (%)",
)
apply_notebook_style(skip_figure, height=CHART_HEIGHTS["time_series"])

plug_in_skip_median=0.12  plug_in_day_factor_sd=0.50  plug_in_week_factor_sd=0.17  (read live from assumptions.SHARED_FACTORS)


In [12]:
pd.DataFrame(100 * study_skip_share, columns=study_dates).round(1).rename_axis("world_id")

,2026-01-12,2026-01-13,2026-01-14,2026-01-15,2026-01-16,2026-01-17,2026-01-18
world_id,,,,,,,
0,18.7,26.2,15.7,13.2,21.7,11.1,24.3
1,17.2,9.1,28.6,14.1,8.3,9.3,24.7
2,7.8,15.1,20.4,10.0,9.5,13.5,8.2
3,7.9,11.5,17.0,14.5,14.0,14.6,18.4
4,12.6,12.5,8.3,15.2,5.9,25.7,17.6


### Holidays shift the skip share only

`clock.holiday_flags` marks the first Monday study evening and every study evening respectively;
neither switch changes a trip, a clock or a draw — only the skip share a plug uniform is compared
against (lead decision, 29 September 2026).


In [13]:
flags_off = clock.holiday_flags(settings, bank_holiday_monday=False, half_term_week=False)
flags_bank_holiday = clock.holiday_flags(settings, bank_holiday_monday=True, half_term_week=False)
flags_half_term = clock.holiday_flags(settings, bank_holiday_monday=False, half_term_week=True)

study_slice = slice(warmup_day_count, warmup_day_count + settings.study_days)
assert not flags_off["holiday"].any()
assert int(flags_bank_holiday["holiday"][study_slice].sum()) == 1, (
    "exactly one Monday study evening"
)
assert int(flags_half_term["holiday"][study_slice].sum()) == settings.study_days
assert not flags_bank_holiday["holiday"][:warmup_day_count].any(), "holidays never touch warm-up"
shift_on_holiday = flags_bank_holiday["skip_logit_shift"][flags_bank_holiday["holiday"]]
assert np.allclose(shift_on_holiday, assumptions.HOLIDAY_SKIP_LOGIT_SHIFT)
print(
    f"bank_holiday_monday marks 1 study evening; half_term_week marks all {settings.study_days}; "
    f"both use the same +{assumptions.HOLIDAY_SKIP_LOGIT_SHIFT:g} skip-logit shift (read live), "
    "read from clock.holiday_flags, not restated."
)

bank_holiday_monday marks 1 study evening; half_term_week marks all 7; both use the same +0.3 skip-logit shift (read live), read from clock.holiday_flags, not restated.


### The always-plugged cohort is excluded from the skip share

Decision 0004 item 60 and contract §10.1b are explicit: `always_plugged_in`'s one session covers
the whole horizon, so it is excluded from the skip share, not merely unaffected by it in practice.
The toy 200-EV population from section 1 has `always_plugged_in` EVs; feeding an extreme skip share
(near-certain skip for every clocked EV) into `sample_connection_opportunities` shows the always-
plugged cohort's sessions stay accepted while the clocked cohorts collapse.


In [14]:
toy_departure_rng = np.random.default_rng(SEED + 1)
toy_departures = sampling.sample_departure_times(
    toy_settings,
    TOY_UNITS,
    toy_departure_rng,
    scale_minutes_by_cohort={c: v["departure_scale_minutes"] for c, v in by_cohort.items()},
    t_df=t_df,
    clip_minutes=clip_minutes,
)
extreme_skip = np.full((toy_settings.evaluation_world_count, toy_settings.sampled_day_count), 0.999)
toy_accepted, _, _ = sampling.sample_connection_opportunities(
    toy_settings,
    TOY_UNITS,
    toy_departure_rng,
    departure_utc=toy_departures,
    # Every toy EV drives every day, so each night is its own would-be session (item 68 otherwise
    # keeps a non-driving EV plugged in and its later plug draws unused).
    drives_today=np.ones(
        (toy_settings.evaluation_world_count, toy_settings.sampled_day_count, len(TOY_UNITS)),
        dtype=bool,
    ),
    plug_in_scale_minutes_by_cohort={c: v["plug_in_scale_minutes"] for c, v in by_cohort.items()},
    weekend_plug_in_scale_minutes_by_cohort={
        c: v["weekend_plug_in_scale_minutes"] for c, v in by_cohort.items()
    },
    t_df=t_df,
    clip_minutes=clip_minutes,
    skip_share=extreme_skip,
)
always_mask = (TOY_UNITS["cohort_id"] == "always_plugged_in").to_numpy()
clocked_mask = ~always_mask
assert always_mask.any(), "the toy population must include always_plugged_in EVs"
# Always-plugged EVs have one session covering the whole horizon, recorded on the first sampled
# date only (sampling.sample_connection_opportunities); it is never re-drawn on a later date.
assert bool(toy_accepted[:, 0, always_mask].all()), (
    "always-plugged's one session ignores the skip share"
)
assert not bool(toy_accepted[:, 1:, always_mask].any()), (
    "always-plugged has no session on later dates"
)
clocked_acceptance = float(toy_accepted[:, :, clocked_mask].mean())
assert clocked_acceptance < 0.05, "a 99.9% skip share should collapse clocked-cohort acceptance"
print(
    f"always_plugged_in ({int(always_mask.sum())} EVs): its one session stays accepted even at a "
    f"99.9% skip share. Clocked cohorts: accepted {100 * clocked_acceptance:.2f}% of would-be "
    "sessions."
)

always_plugged_in (2 EVs): its one session stays accepted even at a 99.9% skip share. Clocked cohorts: accepted 0.09% of would-be sessions.


## 4. Charger manufacturers, response rates and outages

Four illustrative manufacturers (no real brand, like the network zones) each carry a share of the
fleet, a response rate (the share of their sessions that follow a smart plan on a normal night) and
a per-night cloud-outage probability that removes their whole slice from control for that night
(decision 0004 item 59; contract §10.1d). Assignment is the population's last draw — independent of
cohort, zone and control group — so editing a manufacturer's response rate or outage probability
moves no other draw (demonstrated in section 5).

The kernel reads this directly (contract §10.1e, lane J1b, merged `43e7fed`):
`session_non_response_probability` becomes `SmartCharging.non_response`, and the per-archetype
`behaviour.non_response_probability.<cohort_id>` records §9.5 used are retired (there is no longer
a `non_response_probability_by_cohort` argument anywhere in `forecast.py`). A session that ignores
its plan keeps the plan on record — the aggregator does not know in advance who will ignore it —
but is dispatched by the *normal* rule, full power from plug-in, for its whole session;
`physics.plan_status` codes which reason applied (`PLAN_FOLLOWS`, `BASE_NON_RESPONSE`,
`MAKER_OUTAGE`, `CONTROL_OUTAGE_EVENT`, `CONTROL_GROUP`).


In [15]:
manufacturer_table = pd.DataFrame(
    [
        {
            "manufacturer_id": maker_id,
            "label": MANUFACTURER_LABELS[maker_id],
            "share": factor_values[f"manufacturers.share.{maker_id}"],
            "response_rate": factor_values[f"manufacturers.response_rate.{maker_id}"],
            "outage_probability_per_night": factor_values[
                f"manufacturers.outage_probability_per_night.{maker_id}"
            ],
        }
        for maker_id in MANUFACTURER_IDS
    ]
).set_index("manufacturer_id")
manufacturer_counts = BASE.units["manufacturer_id"].value_counts().reindex(MANUFACTURER_IDS)
manufacturer_table["fleet_count"] = manufacturer_counts
assert int(manufacturer_table["fleet_count"].sum()) == VEHICLE_COUNT
manufacturer_table

,label,share,response_rate,outage_probability_per_night,fleet_count
manufacturer_id,,,,,
m1,Maker A,0.25,0.95,0.02,13
m2,Maker B,0.25,0.95,0.02,13
m3,Maker C,0.25,0.95,0.02,12
m4,Maker D,0.25,0.95,0.02,12


### Non-response, per session: a maker outage forces it, otherwise it is the maker's response rate

`session_non_response_probability` (world, study night, EV) is a §10.1 data product on every
action-model `SimulatedForecast`: 1.0 for a control EV or on its maker's outage night, `1 −
response_rate` otherwise. It is also, now, exactly what the smart-charging kernel reads for
`SmartCharging.non_response` (§10.1e; the per-cohort array it used before is retired).

To see the mechanism without waiting for a random outage to land in a 5-week run, this section
deliberately forces one manufacturer's outage probability to 1.0 — every one of its sessions ignores
its plan, every study night, by construction (the same edge case contract §10.1d's own tests use).


In [16]:
forced_maker = "m2"
outage_factor_values = dict(factor_values)
outage_factor_values[f"manufacturers.outage_probability_per_night.{forced_maker}"] = 1.0
OUTAGE_DEMO = simulate_forecast(
    settings,
    fixture,
    model="action",
    **{**inputs, "shared_factor_assumptions": outage_factor_values},
)

maker_index_of = dict(zip(MANUFACTURER_IDS, range(len(MANUFACTURER_IDS)), strict=True))
forced_index = maker_index_of[forced_maker]
other_indices = [i for i in range(len(MANUFACTURER_IDS)) if i != forced_index]
outage_array = OUTAGE_DEMO.manufacturer_outage  # (world, study night, maker)
assert bool(outage_array[:, :, forced_index].all()), (
    "the forced maker must be out every study night"
)
# Forcing one maker's outage probability must not move another maker's own outage draw: same
# uniforms, same unchanged threshold (§10.1d: one block, whatever the probabilities).
assert np.array_equal(
    outage_array[:, :, other_indices], BASE.manufacturer_outage[:, :, other_indices]
)

non_response = OUTAGE_DEMO.session_non_response_probability  # (world, study night, EV)
response_rate = np.array(
    [factor_values[f"manufacturers.response_rate.{m}"] for m in MANUFACTURER_IDS]
)
maker_index_by_ev = OUTAGE_DEMO.units["manufacturer_id"].map(maker_index_of).to_numpy()
# Per maker, whatever nights it happens to be out (the forced one every night, any other maker
# only if the default ~2%/night probability happened to fire): non-response is 1.0 on an outage
# night and exactly 1 - response_rate otherwise, never a blend of the two (§10.1e).
for maker_id in MANUFACTURER_IDS:
    idx = maker_index_of[maker_id]
    maker_evs = maker_index_by_ev == idx
    outaged_nights = outage_array[:, :, idx][:, :, np.newaxis]  # (world, night, 1), broadcasts
    expected = np.where(outaged_nights, 1.0, 1.0 - response_rate[idx])
    assert np.allclose(non_response[:, :, maker_evs], expected), (
        f"{maker_id}'s non-response must be 1.0 on its outage nights, 1 - response_rate otherwise"
    )
forced_evs = maker_index_by_ev == forced_index

mean_non_response = pd.Series(
    {
        maker_id: float(non_response[:, :, maker_index_by_ev == maker_index_of[maker_id]].mean())
        for maker_id in MANUFACTURER_IDS
    },
    name="mean session_non_response_probability",
)
print(f"{forced_maker} ({MANUFACTURER_LABELS[forced_maker]}) forced to outage_probability=1.0.")
mean_non_response.to_frame()

m2 (Maker B) forced to outage_probability=1.0.


,mean session_non_response_probability
m1,0.077143
m2,1.000000
m3,0.077143
m4,0.050000


In [17]:
bar_figure = go.Figure(
    go.Bar(
        x=[MANUFACTURER_LABELS[m] for m in MANUFACTURER_IDS],
        y=mean_non_response.reindex(MANUFACTURER_IDS).to_numpy(),
        marker_color=list(ARCHETYPE_COLOURS[:4]),
    )
)
bar_figure.update_layout(
    title_text=f"Mean session non-response probability by manufacturer ({forced_maker} forced out)",
    yaxis_title="probability",
)
apply_notebook_style(bar_figure, height=CHART_HEIGHTS["histogram"])

### Now it changes charging: the forced maker's EVs run the normal rule

`physics.simulate_unit_intervals` runs the same kernel as the fleet aggregator but keeps each EV
separate, so it can show *which* sessions ran which rule. On the selected path, every interval
whose `plan_status` is `MAKER_OUTAGE` should import exactly what the *normal* (unmanaged) path
imports for that EV and interval — the normal rule, not a coincidence. Every other manufacturer's
EVs, untouched by the forced outage, should see no change against `BASE` at all.


In [18]:
def _unit_intervals(result, *, smart: bool) -> dict[str, np.ndarray]:
    return physics.simulate_unit_intervals(
        settings,
        result.units,
        result.evaluation.inputs,
        public_top_up=result.public_top_up,
        effective_efficiency_miles_per_battery_kwh=(
            result.evaluation.effective_efficiency_miles_per_battery_kwh
        ),
        smart_charging=result.smart_charging if smart else None,
    )


outage_selected = _unit_intervals(OUTAGE_DEMO, smart=True)
outage_normal = _unit_intervals(OUTAGE_DEMO, smart=False)
base_selected = _unit_intervals(BASE, smart=True)

# Every interval a maker outage forced (§10.1e): the selected path must import exactly what the
# normal rule would have, not merely "less" or "differently".
maker_outage_cells = outage_selected["plan_status"] == physics.MAKER_OUTAGE
maker_outage_cell_count = int(maker_outage_cells.sum())
assert maker_outage_cell_count > 0, "the forced maker must produce some MAKER_OUTAGE intervals"
maker_outage_dispatches_normally = bool(
    np.allclose(
        outage_selected["home_grid_import_kwh"][maker_outage_cells],
        outage_normal["home_grid_import_kwh"][maker_outage_cells],
    )
)
assert maker_outage_dispatches_normally, (
    "a MAKER_OUTAGE interval must import exactly what the normal rule would"
)

# Another maker's EVs: same population, same non_response array (checked in section 5) -- their
# realised charging must be untouched by forcing m2's outage.
other_evs = maker_index_by_ev != forced_index
other_makers_unchanged = bool(
    np.allclose(
        outage_selected["home_grid_import_kwh"][:, :, other_evs],
        base_selected["home_grid_import_kwh"][:, :, other_evs],
    )
)
assert other_makers_unchanged, "another maker's EVs must see no change from forcing m2's outage"
forced_import_changed = not np.allclose(
    outage_selected["home_grid_import_kwh"][:, :, forced_evs],
    base_selected["home_grid_import_kwh"][:, :, forced_evs],
)
assert forced_import_changed, "forcing the outage must change the forced maker's own charging"

maker_outage_share = pd.Series(
    {
        maker_id: float(
            (
                outage_selected["plan_status"][:, :, maker_index_by_ev == maker_index_of[maker_id]]
                == physics.MAKER_OUTAGE
            ).mean()
        )
        for maker_id in MANUFACTURER_IDS
    },
    name="share of study intervals with plan_status == MAKER_OUTAGE",
)
print(
    f"{maker_outage_cell_count} (world, interval, EV) cells ran under a maker outage; every one "
    "imported exactly what the normal rule would."
)
maker_outage_share.to_frame()

11260 (world, interval, EV) cells ran under a maker outage; every one imported exactly what the normal rule would.


,share of study intervals with plan_status == MAKER_OUTAGE
m1,0.020788
m2,0.482738
m3,0.013046
m4,0.000000


In [19]:
status_figure = go.Figure(
    go.Bar(
        x=[MANUFACTURER_LABELS[m] for m in MANUFACTURER_IDS],
        y=maker_outage_share.reindex(MANUFACTURER_IDS).to_numpy(),
        marker_color=list(ARCHETYPE_COLOURS[:4]),
    )
)
status_figure.update_layout(
    title_text=f"Share of study intervals dispatched under MAKER_OUTAGE ({forced_maker} forced)",
    yaxis_title="share of intervals",
)
apply_notebook_style(status_figure, height=CHART_HEIGHTS["histogram"])

## 5. Common random numbers: editing one record moves only its own channel

Plan M6: every sampler consumes a fixed number of draws whatever the parameter values, in a fixed
order, so editing one assumption record changes only the outputs its own formula touches and moves
the RNG position seen by nothing else. Two comparisons against `BASE`, same seed and settings
throughout:

- `OUTAGE_DEMO` (above) edits a manufacturer's outage probability — the **last** draw of the run.
- `DAY_SD_DEMO` below doubles the (live) day-factor SD — a channel drawn well before manufacturer
  outages, but still only its own.


In [20]:
day_sd_factor_values = dict(factor_values)
day_sd_factor_values["behaviour.plug_in_day_factor_sd"] = 2.0 * day_sd
DAY_SD_DEMO = simulate_forecast(
    settings,
    fixture,
    model="action",
    **{**inputs, "shared_factor_assumptions": day_sd_factor_values},
)


def _datetimes_equal(a: np.ndarray, b: np.ndarray) -> bool:
    # np.array_equal treats NaT != NaT (like NaN), so it reports "different" for two identical
    # arrays that both hold NaT on a non-driving day. Comparing the int64 view instead compares
    # the raw bit pattern, so two NaTs (the same fixed bit pattern) compare equal.
    return bool(np.array_equal(np.asarray(a).astype("int64"), np.asarray(b).astype("int64")))


base_inputs = BASE.evaluation.inputs
demo_inputs = DAY_SD_DEMO.evaluation.inputs
crn_checks = {
    "OUTAGE_DEMO: population (cohort, zone, manufacturer) identical to BASE": bool(
        BASE.units[["cohort_id", "zone_id", "manufacturer_id"]].equals(
            OUTAGE_DEMO.units[["cohort_id", "zone_id", "manufacturer_id"]]
        )
    ),
    "OUTAGE_DEMO: departures and trips identical to BASE": (
        _datetimes_equal(
            base_inputs["trip_departure_utc"], OUTAGE_DEMO.evaluation.inputs["trip_departure_utc"]
        )
        and np.array_equal(
            base_inputs["drives_today"], OUTAGE_DEMO.evaluation.inputs["drives_today"]
        )
    ),
    "OUTAGE_DEMO: skip share identical to BASE": bool(
        np.array_equal(
            BASE.plug_in_factors["skip_share"], OUTAGE_DEMO.plug_in_factors["skip_share"]
        )
    ),
    "OUTAGE_DEMO: manufacturer_outage differs from BASE": not bool(
        np.array_equal(BASE.manufacturer_outage, OUTAGE_DEMO.manufacturer_outage)
    ),
    "OUTAGE_DEMO: another maker's realised charging is unchanged": other_makers_unchanged,
    "OUTAGE_DEMO: the forced maker's own realised charging changes (the kernel reads it now)": (
        forced_import_changed
    ),
    "DAY_SD_DEMO: population identical to BASE": bool(
        BASE.units[["cohort_id", "zone_id", "manufacturer_id"]].equals(
            DAY_SD_DEMO.units[["cohort_id", "zone_id", "manufacturer_id"]]
        )
    ),
    "DAY_SD_DEMO: departures and trips identical to BASE (drawn before the skip factors)": (
        _datetimes_equal(base_inputs["trip_departure_utc"], demo_inputs["trip_departure_utc"])
        and np.array_equal(base_inputs["drives_today"], demo_inputs["drives_today"])
        and np.array_equal(base_inputs["outbound_miles"], demo_inputs["outbound_miles"])
    ),
    "DAY_SD_DEMO: week/day normal draws identical (only the day_sd multiplier changed)": bool(
        np.array_equal(BASE.plug_in_factors["week"], DAY_SD_DEMO.plug_in_factors["week"])
        and np.array_equal(BASE.plug_in_factors["day"], DAY_SD_DEMO.plug_in_factors["day"])
    ),
    "DAY_SD_DEMO: skip share differs from BASE": not bool(
        np.array_equal(
            BASE.plug_in_factors["skip_share"], DAY_SD_DEMO.plug_in_factors["skip_share"]
        )
    ),
    "DAY_SD_DEMO: manufacturer_outage identical to BASE (drawn after, fixed draw count)": bool(
        np.array_equal(BASE.manufacturer_outage, DAY_SD_DEMO.manufacturer_outage)
    ),
}
for description, passed in crn_checks.items():
    assert passed, description
pd.DataFrame({"check": crn_checks.keys(), "passed": crn_checks.values()})

,check,passed
0,"OUTAGE_DEMO: population (cohort, zone, manufac...",True
1,OUTAGE_DEMO: departures and trips identical to...,True
2,OUTAGE_DEMO: skip share identical to BASE,True
3,OUTAGE_DEMO: manufacturer_outage differs from ...,True
4,OUTAGE_DEMO: another maker's realised charging...,True
5,OUTAGE_DEMO: the forced maker's own realised c...,True
6,DAY_SD_DEMO: population identical to BASE,True
7,DAY_SD_DEMO: departures and trips identical to...,True
8,DAY_SD_DEMO: week/day normal draws identical (...,True
9,DAY_SD_DEMO: skip share differs from BASE,True


## 6. Law of large numbers: shared factors don't diversify away

Contract §10.0's point, directly: if every EV's plug-in were independent, the fleet's *relative*
spread (SD ÷ mean) of a night's plug-in count would shrink as `1/√N`. With the shared day and week
factors switched on (the run's own default), every EV's odds move together, so part of that spread
never diversifies away, however large the fleet. The sweep below draws the shared factors once per
simulated week (`sample_plug_in_factors`, `plug_in_skip_share` — the real functions, at their live
default SDs) and then compares, at growing fleet sizes, the *same* environment applied to more EVs.


In [21]:
lln_rng = np.random.default_rng(SEED)
SWEEP_WORLDS = 300
FLEET_SIZES = (50, 200, 1_000, 5_000, 20_000)
P_PLUG = 1.0  # a p=1 clocked cohort (five of the six source cohorts)

sweep_week, sweep_day = sampling.sample_plug_in_factors(lln_rng, SWEEP_WORLDS, 1)
sweep_skip = sampling.plug_in_skip_share(
    sweep_week, sweep_day, np.zeros(1), median=skip_median, day_sd=day_sd, week_sd=week_sd
)[:, 0]
sweep_effective = P_PLUG * (1.0 - sweep_skip)  # (SWEEP_WORLDS,)

rows = []
for fleet_size in FLEET_SIZES:
    draw_rng = np.random.default_rng(SEED + fleet_size)
    counts = (draw_rng.random((SWEEP_WORLDS, fleet_size)) < sweep_effective[:, None]).sum(axis=1)
    mean_count = float(counts.mean())
    relative_sd = float(counts.std() / mean_count)
    independent_relative_sd = float(
        np.sqrt((mean_count / fleet_size) * (1 - mean_count / fleet_size) / fleet_size)
    )
    rows.append(
        {
            "fleet_size": fleet_size,
            "relative_sd_actual": relative_sd,
            "relative_sd_if_independent": independent_relative_sd,
        }
    )
sweep_table = pd.DataFrame(rows).set_index("fleet_size")

# The theoretical asymptote (N -> infinity): relative SD of any skip-driven share is
# SD(p_eff) / E[p_eff], estimated from a large, independent Monte Carlo draw of the same real
# functions, not a hand-derived formula.
asymptote_rng = np.random.default_rng(SEED + 1)
big_week, big_day = sampling.sample_plug_in_factors(asymptote_rng, 200_000, 1)
big_skip = sampling.plug_in_skip_share(
    big_week, big_day, np.zeros(1), median=skip_median, day_sd=day_sd, week_sd=week_sd
)[:, 0]
big_p_eff = P_PLUG * (1.0 - big_skip)
asymptote = float(big_p_eff.std() / big_p_eff.mean())
sweep_table

,relative_sd_actual,relative_sd_if_independent
fleet_size,,
50,0.082443,0.047331
200,0.071702,0.023847
1000,0.069132,0.010639
5000,0.068193,0.004759
20000,0.068132,0.002376


In [22]:
lln_figure = go.Figure()
lln_figure.add_trace(
    go.Scatter(
        x=list(FLEET_SIZES),
        y=sweep_table["relative_sd_actual"],
        mode="lines+markers",
        name="actual (shared day/week factor on)",
        line={"color": ARCHETYPE_COLOURS[3], "width": 2.5},
    )
)
lln_figure.add_trace(
    go.Scatter(
        x=list(FLEET_SIZES),
        y=sweep_table["relative_sd_if_independent"],
        mode="lines+markers",
        name="if independent (1/√N)",
        line={"color": ARCHETYPE_COLOURS[0], "width": 1.5, "dash": "dot"},
    )
)
lln_figure.add_hline(
    y=asymptote,
    line_dash="dash",
    line_color=ARCHETYPE_COLOURS[3],
    annotation_text="asymptote (N → ∞)",
)
lln_figure.update_layout(
    title_text="Fleet plug-in count: relative spread levels off, it does not keep shrinking",
    xaxis_title="fleet size (EVs)",
    yaxis_title="relative SD of the night's plug-in count",
    xaxis_type="log",
)
apply_notebook_style(lln_figure, height=CHART_HEIGHTS["time_series"])

In [23]:
relative_sd = sweep_table["relative_sd_actual"].to_numpy()
independent_relative_sd = sweep_table["relative_sd_if_independent"].to_numpy()
fleet_sizes = np.array(FLEET_SIZES, dtype=float)

lln_checks = {
    "relative SD never rises with fleet size (within sampling noise)": bool(
        np.all(np.diff(relative_sd) <= 0.15 * relative_sd[:-1])
    ),
    "relative SD shrinks far less than 1/sqrt(N) alone, smallest to largest fleet size": (
        relative_sd[-1] / relative_sd[0]
        > 3 * (independent_relative_sd[-1] / independent_relative_sd[0])
    ),
    "the largest fleet's relative SD sits within 25% of the analytic N -> infinity asymptote": (
        abs(relative_sd[-1] / asymptote - 1) < 0.25
    ),
    "the independent (1/sqrt(N)) reference keeps shrinking all the way to the largest fleet": bool(
        independent_relative_sd[-1] < 0.2 * independent_relative_sd[0]
    ),
}
for description, passed in lln_checks.items():
    assert passed, description
pd.DataFrame({"check": lln_checks.keys(), "passed": lln_checks.values()})

,check,passed
0,relative SD never rises with fleet size (withi...,True
1,relative SD shrinks far less than 1/sqrt(N) al...,True
2,the largest fleet's relative SD sits within 25...,True
3,the independent (1/sqrt(N)) reference keeps sh...,True


## 7. Sanity checks


In [24]:
final_checks = {
    "every cohort count at N=50 sums to the fleet size": int(base_counts.sum()) == VEHICLE_COUNT,
    "always_plugged_in never skips, at a near-certain skip share": bool(
        toy_accepted[:, 0, always_mask].all()
    ),
    "plug-in and departure shifts are uncorrelated (|r| < 0.08)": abs(correlation) < 0.08,
    "manufacturer counts sum to the fleet size": int(manufacturer_table["fleet_count"].sum())
    == VEHICLE_COUNT,
    "a forced maker outage drives that maker's non-response to exactly 1.0": bool(
        np.allclose(non_response[:, :, forced_evs], 1.0)
    ),
    "a MAKER_OUTAGE interval always charges by the normal rule, exactly": (
        maker_outage_cell_count > 0 and maker_outage_dispatches_normally
    ),
    "forcing one maker's outage changes only that maker's realised charging": (
        other_makers_unchanged and forced_import_changed
    ),
    "common random numbers hold for both the outage and day-SD edits": all(crn_checks.values()),
    "the fleet-size sweep shows levelling off, not 1/sqrt(N) shrinkage": all(lln_checks.values()),
}
for description, passed in final_checks.items():
    assert passed, description
pd.DataFrame({"check": final_checks.keys(), "passed": final_checks.values()})

,check,passed
0,every cohort count at N=50 sums to the fleet size,True
1,"always_plugged_in never skips, at a near-certa...",True
2,plug-in and departure shifts are uncorrelated ...,True
3,manufacturer counts sum to the fleet size,True
4,a forced maker outage drives that maker's non-...,True
5,a MAKER_OUTAGE interval always charges by the ...,True
6,forcing one maker's outage changes only that m...,True
7,common random numbers hold for both the outage...,True
8,"the fleet-size sweep shows levelling off, not ...",True


## 8. Limitations

- **Illustrative and synthetic only.** Every clock scale, skip-share default, manufacturer share,
  response rate and outage probability is a labelled, editable illustrative assumption
  (decision 0004 items 51, 59, 60, 63 and 64); none is fitted to observed telemetry.
- **Small run.** 50 EVs and 5 simulated weeks keep this notebook fast; the app's default run (1,000
  EVs, 100 simulated weeks) gives steadier bands. Section 6's fleet-size sweep uses the shared-factor
  functions directly (no physics kernel) so it can go far larger, cheaply.
- **Shared timing variation is not modelled.** People plugging in earlier or later together on some
  days — as opposed to more or fewer of them plugging in — is a stated limitation of the skip-share
  design (decision 0004 item 64).
- **The fleet-size sweep isolates one mechanism.** It uses a single `p = 1` clocked cohort (five of
  the six source cohorts share that plug probability) and only the day/week skip factors, with a
  plain Bernoulli draw standing in for a real fleet's connections. The real
  `availability.firm_share_by_fleet_size` (contract §10.2c, built on a `run_forecast` result, not
  the raw `simulate_forecast` this notebook mostly reads) also carries manufacturer outages and the
  tariff-synchronisation (herding) factor already visible in notebook 04 — this sweep's point is
  the mechanism, not that frame's own number.
